# CIFOR Global Wetlands V3 — test clip

Gumbricht et al. (2017) tropical/subtropical wetlands, viewer: <https://www2.cifor.org/global-wetlands/>.

The native GeoTIFF is [doi:10.17528/CIFOR/DATA.00058](https://doi.org/10.17528/CIFOR/DATA.00058) (`TROP-SUBTROP_WetlandV3b_2016_CIFOR.7z`, 231 m). `data.cifor.org` currently drops the TLS handshake, so `georg_dev/download_cifor_wetlands.py` reads the public tiled map the viewer uses and writes class codes (not RGB).

Pixel values: 0 nodata, 10 open water, 20 mangrove, 30 swamp, 40 fen, 50 riverine/lacustrine, 60 floodout, 70 floodplain, 80 marsh, 90 arid marsh, 100 wet meadow.

Output is EPSG:4326, nearest-neighbour, cache resolution ~153 m (level 10).

In [ ]:
from pathlib import Path
import sys

import rasterio
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "georg_dev" / "download_cifor_wetlands.py").exists():
    ROOT = Path.cwd() / "topics" / "11-B-FLOODED-FOREST"
sys.path.insert(0, str(ROOT / "georg_dev"))

import download_cifor_wetlands as cifor

# west, south, east, north
BBOX = (
    -49.1513175964354971,
    -2.2793788909912100,
    -48.7919921875000000,
    -1.7499319314956701,
)
out = cifor.download_clip(bbox=BBOX, level=10)

with rasterio.open(out) as ds:
    arr = ds.read(1)
    print(ds.crs, ds.bounds)
codes, counts = np.unique(arr, return_counts=True)
names = {v: k for k, v in cifor.CLASS_CODE.items()}
for code, n in zip(codes, counts):
    label = "nodata" if code == 0 else names.get(int(code), "?")
    print(f"{int(code):3d}  {label:28s}  {int(n):6d} px")
